In [ ]:
import json
import requests
import pandas as pd

url = "https://raw.githubusercontent.com/amanb2000/Emo_LLM/main/datasets/4oautism_promptgen_20240617.jsonl"
response = requests.get(url, timeout=30)
response.raise_for_status()

records = []
malformed_lines = 0
for line_number, line in enumerate(response.text.splitlines(), start=1):
    if not line.strip():
        continue
    try:
        records.append(json.loads(line))
    except json.JSONDecodeError:
        malformed_lines += 1

if not records:
    raise ValueError("The public JSONL dataset contained no valid records.")

autism_df = pd.DataFrame(records)
required_columns = {"prompt", "tag"}
missing_columns = required_columns.difference(autism_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Loaded {len(autism_df)} valid public records; skipped {malformed_lines} malformed lines.")
print("Prompt-tag counts:")
print(autism_df["tag"].value_counts())

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [word for word in text.split() if word not in stop_words]
    return " ".join(words)

autism_df["cleaned_prompt"] = autism_df["prompt"].map(preprocess_text)
print("Prepared public prompt text for the tag-classification demo.")

## Public prompt-tag demo

This model predicts labels attached to public prompt text. It does not detect or diagnose autism, depression, or any health condition.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

labels = autism_df["tag"].astype(str)
train_text, test_text, y_train_autism, y_test_autism = train_test_split(
    autism_df["cleaned_prompt"],
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_autism = tfidf_vectorizer.fit_transform(train_text)
X_test_autism = tfidf_vectorizer.transform(test_text)
print("Train/test split completed before fitting TF-IDF.")

In [ ]:
print("Training feature shape:", X_train_autism.shape)
print("Testing feature shape:", X_test_autism.shape)
print("Training label count:", y_train_autism.shape[0])
print("Testing label count:", y_test_autism.shape[0])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

autism_model = LogisticRegression(max_iter=1000, solver="liblinear")
autism_model.fit(X_train_autism, y_train_autism)
y_pred_autism = autism_model.predict(X_test_autism)

print("Prompt-tag classification report (not a health assessment):")
print(classification_report(y_test_autism, y_pred_autism, zero_division=0))
print("Weighted F1:", f1_score(y_test_autism, y_pred_autism, average="weighted"))

In [ ]:
!pip install -q pymongo
import os
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords

try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def create_mongo_client():
    try:
        from google.colab import userdata
    except ImportError:
        mongo_uri = os.environ.get("MONGODB_URI")
    else:
        mongo_uri = userdata.get("MONGODB_URI")

    if not mongo_uri:
        raise RuntimeError(
            "Set MONGODB_URI in Colab Secrets or the MONGODB_URI environment variable."
        )

    from pymongo import MongoClient
    return MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(word for word in text.split() if word not in stop_words)

def responses_to_text(value):
    if isinstance(value, (list, tuple)):
        if value and isinstance(value[0], dict):
            return " ".join(str(item.get("selected_text", "")) for item in value)
        return " ".join(str(item) for item in value)
    if value is None:
        return ""
    return str(value)

client = create_mongo_client()
try:
    client.admin.command("ping")
    collection = client["bdi_result"]["users"]
    projection = {
        "_id": 0,
        "responses": 1,
        "raw_responses_for_ml": 1,
        "risk_level": 1,
    }
    mongo_records = list(collection.find({"research_consent": True}, projection))
finally:
    client.close()

# An empty research dataset is valid for a new installation. The automatic
# pipeline will begin collecting separately consented records after assessments.
df = pd.DataFrame(mongo_records)
if df.empty:
    df = pd.DataFrame(columns=["risk_level", "responses", "raw_responses_for_ml"])
required_columns = {"risk_level"}
missing_columns = required_columns.difference(df.columns)
if missing_columns or not ({"responses", "raw_responses_for_ml"} & set(df.columns)):
    raise ValueError("Consented records are missing response text or risk labels.")

response_column = "raw_responses_for_ml" if "raw_responses_for_ml" in df.columns else "responses"
df["responses_cleaned"] = df[response_column].map(responses_to_text).map(preprocess_text)
df = df.dropna(subset=["risk_level"])
if df.empty:
    raise ValueError("No consented records with risk labels are available for training.")

print(f"Loaded {len(df)} consented records; no personal identifiers were retrieved.")

## Separate the experiments

The public prompt-tag classifier and the BDI-II score-based experiment have different labels and must not be combined. The BDI-II model is a demonstration only, not a diagnosis or clinically validated assessment. Training reads only records whose owners explicitly consented to model use.

### Legacy one-time BDI-II experiment (optional)\nThe original manual train/test/SMOTE cells are preserved below. They are disabled by default because the new MongoDB threshold trainer handles batch training and safely evaluates candidate models. Set `RUN_LEGACY_MANUAL_BDI_TRAINING = True` only if you intentionally want to run the old experiment on the currently available consented records.\n

In [ ]:
RUN_LEGACY_MANUAL_BDI_TRAINING = False


In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.feature_extraction.text import TfidfVectorizer

    if "responses_cleaned" not in df.columns:
        raise ValueError("Run the consented-data loading cell first.")

    X_bdi2 = df["responses_cleaned"].fillna("").astype(str)
    y_bdi2 = df["risk_level"].astype(str)
    print("BDI-II demonstration records:", len(df))
    print("Score-derived label counts:")
    print(y_bdi2.value_counts())

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer

    class_counts = y_bdi2.value_counts()
    if class_counts.min() < 2:
        raise ValueError("Each risk category needs at least two consented records for a stratified split.")

    train_text, test_text, y_train_bdi2, y_test_bdi2 = train_test_split(
        X_bdi2,
        y_bdi2,
        test_size=0.2,
        random_state=42,
        stratify=y_bdi2,
    )
    tfidf_vectorizer_bdi2 = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
    X_train_bdi2 = tfidf_vectorizer_bdi2.fit_transform(train_text)
    X_test_bdi2 = tfidf_vectorizer_bdi2.transform(test_text)
    print("Train/test split completed before fitting BDI-II TF-IDF.")

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.linear_model import LogisticRegression

    bdi2_classifier_model = LogisticRegression(max_iter=2000, solver="saga", random_state=42)
    bdi2_classifier_model.fit(X_train_bdi2, y_train_bdi2)
    print("Baseline BDI-II demonstration model trained.")

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.metrics import classification_report, f1_score

    y_pred_bdi2 = bdi2_classifier_model.predict(X_test_bdi2)
    print("Baseline BDI-II demonstration report (not clinical performance):")
    print(classification_report(y_test_bdi2, y_pred_bdi2, zero_division=0))
    print("Weighted F1:", f1_score(y_test_bdi2, y_pred_bdi2, average="weighted"))

In [ ]:
!pip install imbalanced-learn

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from imblearn.over_sampling import SMOTE
    from collections import Counter

    training_counts = Counter(y_train_bdi2)
    minimum_class_count = min(training_counts.values())
    if minimum_class_count < 2:
        raise ValueError("SMOTE needs at least two training examples in every risk category.")

    smote = SMOTE(random_state=42, k_neighbors=min(5, minimum_class_count - 1))
    X_train_bdi2_resampled, y_train_bdi2_resampled = smote.fit_resample(
        X_train_bdi2, y_train_bdi2
    )
    print("Resampled training labels:", Counter(y_train_bdi2_resampled))

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.linear_model import LogisticRegression

    bdi2_classifier_model_smote = LogisticRegression(
        max_iter=2000,
        solver="saga",
        random_state=42,
    )
    bdi2_classifier_model_smote.fit(X_train_bdi2_resampled, y_train_bdi2_resampled)
    print("SMOTE BDI-II demonstration model trained.")

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.metrics import classification_report, f1_score

    y_pred_bdi2_smote = bdi2_classifier_model_smote.predict(X_test_bdi2)
    print("SMOTE BDI-II demonstration report (not clinical performance):")
    print(classification_report(y_test_bdi2, y_pred_bdi2_smote, zero_division=0))
    print("Weighted F1:", f1_score(y_test_bdi2, y_pred_bdi2_smote, average="weighted"))

In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    import joblib

    joblib.dump(bdi2_classifier_model_smote, "lr_bdi2_model_smote.joblib")
    joblib.dump(tfidf_vectorizer_bdi2, "tfidf_vectorizer_bdi2.joblib")
    print("Saved the demonstration model and its fitted vectorizer.")

In [ ]:

import re
from nltk.corpus import stopwords

loaded_lr_model = None
loaded_tfidf_vectorizer = None

def preprocess_text_for_prediction(text):
    text = re.sub(r"[^a-z\s]", " ", str(text).lower())
    try:
        words_to_remove = stop_words
    except NameError:
        words_to_remove = set(stopwords.words("english"))
    return " ".join(word for word in text.split() if word not in words_to_remove)

if RUN_LEGACY_MANUAL_BDI_TRAINING:
    loaded_lr_model = bdi2_classifier_model_smote
    loaded_tfidf_vectorizer = tfidf_vectorizer_bdi2
    example_text = "I have been feeling low and finding it difficult to focus."
    example_vector = loaded_tfidf_vectorizer.transform(
        [preprocess_text_for_prediction(example_text)]
    )
    example_prediction = loaded_lr_model.predict(example_vector)[0]
    print("Illustrative model output only; not a diagnosis:", example_prediction)
else:
    print("Legacy manual classifier is disabled. Use the approved MongoDB champion, if one exists.")


In [ ]:
if RUN_LEGACY_MANUAL_BDI_TRAINING:
    from sklearn.metrics import confusion_matrix

    confusion = confusion_matrix(y_test_bdi2, y_pred_bdi2_smote, labels=bdi2_classifier_model_smote.classes_)
    print("Confusion matrix (rows=true, columns=predicted):")
    print(confusion)
    print("Misclassified test rows:", int((y_test_bdi2.to_numpy() != y_pred_bdi2_smote).sum()))
    print("Individual response text is intentionally not displayed.")

## Automatic MongoDB training and safe model registry

This section adds threshold-based batch training. Assessment storage consent and research/model-training consent are separate. Only records with `research_consent=True` are eligible. The experimental text classifier learns to reproduce score-derived BDI-II categories; it is not a diagnostic model and must not be used for clinical decisions.

**Deployment note:** model artifacts are stored in MongoDB GridFS and metadata in `model_registry`. The notebook must be running when a new assessment triggers the threshold check. For always-on production automation, move `maybe_train_from_mongodb()` into a secured background worker (for example, a scheduled service or MongoDB Change Stream consumer).

In [ ]:

# Install dependencies if running in a fresh Colab/Jupyter environment:
# %pip install -q pymongo scikit-learn joblib

import os
import re
import json
import hashlib
import pickle
from datetime import datetime, timezone
from collections import Counter

import numpy as np
from pymongo import MongoClient, DESCENDING
from gridfs import GridFS
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, accuracy_score

# Tune this threshold to your research protocol. Keep it high enough to avoid
# fitting unstable models on tiny samples. Every class also needs >= 2 records.
MIN_NEW_CONSENTED_RECORDS = 50
TEST_SIZE = 0.20
RANDOM_STATE = 42
MIN_PROMOTION_WEIGHTED_F1 = 0.60  # research-demo gate, not a clinical standard
PROMOTION_MARGIN = 0.00           # candidate must not be worse than champion
MODEL_NAME = "bdi2_text_classifier_experimental"

def _mongo_handles():
    client = create_mongo_client()
    db = client["bdi_result"]
    return client, db, db["users"], db["model_registry"], db["training_runs"]

def _responses_to_text_for_training(value):
    if isinstance(value, (list, tuple)):
        if value and isinstance(value[0], dict):
            return " ".join(str(item.get("selected_text", "")) for item in value)
        return " ".join(str(item) for item in value)
    return "" if value is None else str(value)

def _clean_training_text(value):
    # Keep the same broad preprocessing convention as the existing notebook.
    value = str(value).lower()
    value = re.sub(r"[^a-z\s]", " ", value)
    try:
        words_to_remove = stop_words
    except NameError:
        words_to_remove = set()
    return " ".join(w for w in value.split() if w not in words_to_remove)

def _latest_champion(registry):
    return registry.find_one(
        {"model_name": MODEL_NAME, "status": "champion"},
        sort=[("created_at", DESCENDING)]
    )

def _load_champion_pipeline(db, champion):
    if not champion or not champion.get("gridfs_id"):
        return None
    fs = GridFS(db, collection="model_artifacts")
    payload = fs.get(champion["gridfs_id"]).read()
    return pickle.loads(payload)

def maybe_train_from_mongodb():
    """
    Called after each consented assessment is saved.
    Trains only after MIN_NEW_CONSENTED_RECORDS newly eligible records have
    accumulated since the previous completed training run.
    """
    client, db, users, registry, runs = _mongo_handles()
    try:
        client.admin.command("ping")

        # Use the last completed attempt as the batch boundary. If data is
        # insufficient or invalid, do not advance it; later records can help.
        state = registry.find_one({"_id": f"{MODEL_NAME}:state"})
        last_completed_at = state.get("last_completed_at") if state else None
        new_query = {"research_consent": True}
        if last_completed_at:
            new_query["created_at"] = {"$gt": last_completed_at}

        new_count = users.count_documents(new_query)
        if new_count < MIN_NEW_CONSENTED_RECORDS:
            return (
                f"waiting: {new_count}/{MIN_NEW_CONSENTED_RECORDS} "
                "new research-consented records"
            )

        projection = {
            "_id": 1,
            "responses": 1,
            "raw_responses_for_ml": 1,
            "risk_level": 1,
            "total_score": 1,
            "created_at": 1,
        }
        records = list(users.find(
            {"research_consent": True},
            projection
        ))
        rows = []
        for record in records:
            label = record.get("risk_level")
            response_value = record.get("raw_responses_for_ml") or record.get("responses")
            text_value = _clean_training_text(_responses_to_text_for_training(response_value))
            if label and text_value.strip():
                rows.append({"text": text_value, "label": str(label)})

        if len(rows) < MIN_NEW_CONSENTED_RECORDS:
            return f"waiting: only {len(rows)} usable consented records are available"

        labels = [row["label"] for row in rows]
        class_counts = Counter(labels)
        if len(class_counts) < 2 or min(class_counts.values()) < 2:
            return (
                "waiting: at least two examples per class and at least two classes "
                "are required for a stratified validation split"
            )

        X = [row["text"] for row in rows]
        y = np.asarray(labels)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
        )

        candidate = Pipeline([
            ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
            ("classifier", LogisticRegression(
                max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
            )),
        ])
        candidate.fit(X_train, y_train)
        predictions = candidate.predict(X_test)

        weighted_f1 = float(f1_score(y_test, predictions, average="weighted"))
        macro_f1 = float(f1_score(y_test, predictions, average="macro"))
        accuracy = float(accuracy_score(y_test, predictions))

        champion = _latest_champion(registry)
        champion_pipeline = _load_champion_pipeline(db, champion)
        champion_f1 = None
        if champion_pipeline is not None:
            champion_predictions = champion_pipeline.predict(X_test)
            champion_f1 = float(f1_score(y_test, champion_predictions, average="weighted"))

        passes_quality_gate = weighted_f1 >= MIN_PROMOTION_WEIGHTED_F1
        beats_champion = (
            champion_f1 is None or weighted_f1 >= champion_f1 + PROMOTION_MARGIN
        )
        promote = bool(passes_quality_gate and beats_champion)

        run_id = hashlib.sha256(
            (datetime.now(timezone.utc).isoformat() + str(len(rows))).encode()
        ).hexdigest()[:16]
        version = f"1.{runs.count_documents({'model_name': MODEL_NAME}) + 1}.0"

        # Store the candidate artifact in MongoDB GridFS, even if it is not
        # promoted, so experiments remain auditable and reproducible.
        artifact_bytes = pickle.dumps(candidate, protocol=pickle.HIGHEST_PROTOCOL)
        fs = GridFS(db, collection="model_artifacts")
        gridfs_id = fs.put(
            artifact_bytes,
            filename=f"{MODEL_NAME}_{version}.pkl",
            content_type="application/python-pickle",
            metadata={"model_name": MODEL_NAME, "version": version, "run_id": run_id}
        )

        now = datetime.now(timezone.utc)
        run_doc = {
            "run_id": run_id,
            "model_name": MODEL_NAME,
            "version": version,
            "created_at": now,
            "training_records": len(rows),
            "new_records_since_last_run": new_count,
            "class_counts": dict(class_counts),
            "weighted_f1": weighted_f1,
            "macro_f1": macro_f1,
            "accuracy": accuracy,
            "champion_weighted_f1_on_same_holdout": champion_f1,
            "quality_gate_passed": passes_quality_gate,
            "beats_champion": beats_champion,
            "promoted": promote,
            "gridfs_id": gridfs_id,
            "report": classification_report(
                y_test, predictions, output_dict=True, zero_division=0
            ),
            "validation_note": (
                "Internal holdout evaluation only; labels are derived from BDI-II "
                "scores. Not clinical validation and not suitable for diagnosis."
            ),
        }
        runs.insert_one(run_doc)

        status = "candidate"
        if promote:
            # Atomic status change: keep one active champion for this model name.
            registry.update_many(
                {"model_name": MODEL_NAME, "status": "champion"},
                {"$set": {"status": "superseded", "superseded_at": now}}
            )
            status = "champion"
        registry.insert_one({
            "model_name": MODEL_NAME,
            "version": version,
            "status": status,
            "created_at": now,
            "gridfs_id": gridfs_id,
            "run_id": run_id,
            "weighted_f1": weighted_f1,
            "macro_f1": macro_f1,
            "accuracy": accuracy,
        })

        # Advance the threshold cursor only after a full training/evaluation run.
        registry.update_one(
            {"_id": f"{MODEL_NAME}:state"},
            {"$set": {"last_completed_at": now, "updated_at": now}},
            upsert=True
        )

        print("Candidate evaluation:")
        print(f"  Records: {len(rows)} | New since prior run: {new_count}")
        print(f"  Weighted F1: {weighted_f1:.4f} | Macro F1: {macro_f1:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Champion F1 on same holdout: {champion_f1}")
        print(f"  Model version: {version} | Status: {status}")
        print("  Classification report:")
        print(classification_report(y_test, predictions, zero_division=0))

        if promote:
            return f"promoted version {version} as experimental champion"
        return f"candidate {version} stored but NOT promoted (validation gate not met)"

    finally:
        client.close()

def load_approved_experimental_model():
    """Load only the current champion from MongoDB GridFS."""
    client, db, users, registry, runs = _mongo_handles()
    try:
        champion = _latest_champion(registry)
        if not champion:
            raise RuntimeError("No approved experimental champion exists yet.")
        model = _load_champion_pipeline(db, champion)
        return model, {
            "version": champion.get("version"),
            "weighted_f1": champion.get("weighted_f1"),
            "status": champion.get("status"),
        }
    finally:
        client.close()

def show_model_registry():
    client, db, users, registry, runs = _mongo_handles()
    try:
        return list(registry.find(
            {"model_name": MODEL_NAME},
            {"_id": 0, "model_name": 1, "version": 1, "status": 1,
             "created_at": 1, "weighted_f1": 1, "macro_f1": 1, "accuracy": 1}
        ).sort("created_at", DESCENDING))
    finally:
        client.close()

print("Automatic MongoDB training functions are ready.")
print(f"Training threshold: {MIN_NEW_CONSENTED_RECORDS} new research-consented records.")


In [ ]:
bdi2_questions = [
    {
        "question": "1. Sadness:",
        "options": [
            {"text": "I do not feel sad.", "score": 0},
            {"text": "I feel sad much of the time.", "score": 1},
            {"text": "I am sad all the time.", "score": 2},
            {"text": "I am so sad or unhappy that I can't stand it.", "score": 3}
        ]
    },
    {
        "question": "2. Pessimism:",
        "options": [
            {"text": "I am not discouraged about my future.", "score": 0},
            {"text": "I feel more discouraged about my future than I used to.", "score": 1},
            {"text": "I do not expect things to work out for me.", "score": 2},
            {"text": "I feel my future is hopeless and will only get worse.", "score": 3}
        ]
    },
    {
        "question": "3. Past Failure:",
        "options": [
            {"text": "I do not feel like a failure.", "score": 0},
            {"text": "I have failed more than I should have.", "score": 1},
            {"text": "As I look back, I see a lot of failures.", "score": 2},
            {"text": "I feel I am a total failure as a person.", "score": 3}
        ]
    },
    {
        "question": "4. Loss of Pleasure:",
        "options": [
            {"text": "I get as much pleasure as I ever did from the things I enjoy.", "score": 0},
            {"text": "I don't enjoy things as much as I used to.", "score": 1},
            {"text": "I get very little pleasure from the things I used to enjoy.", "score": 2},
            {"text": "I can't get any pleasure from the things I used to enjoy.", "score": 3}
        ]
    },
    {
        "question": "5. Guilty Feelings:",
        "options": [
            {"text": "I don't feel particularly guilty.", "score": 0},
            {"text": "I feel guilty over many things I have done or should have done.", "score": 1},
            {"text": "I feel quite guilty most of the time.", "score": 2},
            {"text": "I feel guilty all of the time.", "score": 3}
        ]
    },
    {
        "question": "6. Punishment Feelings:",
        "options": [
            {"text": "I don't feel I am being punished.", "score": 0},
            {"text": "I feel I may be punished.", "score": 1},
            {"text": "I expect to be punished.", "score": 2},
            {"text": "I feel I am being punished.", "score": 3}
        ]
    },
    {
        "question": "7. Self-Dislike:",
        "options": [
            {"text": "I feel the same about myself as ever.", "score": 0},
            {"text": "I have lost confidence in myself.", "score": 1},
            {"text": "I am disappointed in myself.", "score": 2},
            {"text": "I dislike myself.", "score": 3}
        ]
    },
    {
        "question": "8. Self-Criticalness:",
        "options": [
            {"text": "I don't criticize or blame myself more than usual.", "score": 0},
            {"text": "I am more critical of myself than I used to be.", "score": 1},
            {"text": "I criticize myself for all of my faults.", "score": 2},
            {"text": "I blame myself for everything bad that happens.", "score": 3}
        ]
    },
    {
        "question": "9. Suicidal Thoughts or Wishes:",
        "options": [
            {"text": "I don't have any thoughts of killing myself.", "score": 0},
            {"text": "I have thoughts of killing myself, but I would not carry them out.", "score": 1},
            {"text": "I would like to kill myself.", "score": 2},
            {"text": "I would kill myself if I had the chance.", "score": 3}
        ]
    },
    {
        "question": "10. Crying:",
        "options": [
            {"text": "I don't cry anymore than I used to.", "score": 0},
            {"text": "I cry more than I used to.", "score": 1},
            {"text": "I cry over every little thing.", "score": 2},
            {"text": "I feel like crying, but I can't.", "score": 3}
        ]
    },
    {
        "question": "11. Agitation:",
        "options": [
            {"text": "I am no more restless or wound up than usual.", "score": 0},
            {"text": "I feel more restless or wound up than usual.", "score": 1},
            {"text": "I am so restless or agitated, it's hard to stay still.", "score": 2},
            {"text": "I am so restless or agitated that I have to keep moving or doing something.", "score": 3}
        ]
    },
    {
        "question": "12. Loss of Interest:",
        "options": [
            {"text": "I have not lost interest in other people or activities.", "score": 0},
            {"text": "I am less interested in other people or things than before.", "score": 1},
            {"text": "I have lost most of my interest in other people or things.", "score": 2},
            {"text": "It's hard to get interested in anything.", "score": 3}
        ]
    },
    {
        "question": "13. Indecisiveness:",
        "options": [
            {"text": "I make decisions about as well as ever.", "score": 0},
            {"text": "I find it more difficult to make decisions than usual.", "score": 1},
            {"text": "I have much greater difficulty in making decisions than I used to.", "score": 2},
            {"text": "I have trouble making any decisions.", "score": 3}
        ]
    },
    {
        "question": "14. Worthlessness:",
        "options": [
            {"text": "I do not feel I am worthless.", "score": 0},
            {"text": "I don't consider myself as worthwhile and useful as I used to.", "score": 1},
            {"text": "I feel more worthless as compared to others.", "score": 2},
            {"text": "I feel utterly worthless.", "score": 3}
        ]
    },
    {
        "question": "15. Loss of Energy:",
        "options": [
            {"text": "I have as much energy as ever.", "score": 0},
            {"text": "I have less energy than I used to have.", "score": 1},
            {"text": "I don't have enough energy to do very much.", "score": 2},
            {"text": "I don't have enough energy to do anything.", "score": 3}
        ]
    },
    {
        "question": "16. Changes in Sleeping Pattern:",
        "options": [
            {"text": "I have not experienced any change in my sleeping.", "score": 0},
            {"text": "I sleep somewhat more than usual. / I sleep somewhat less than usual.", "score": 1},
            {"text": "I sleep a lot more than usual. / I sleep a lot less than usual.", "score": 2},
            {"text": "I sleep most of the day. / I wake up 1-2 hours early and can't get back to sleep.", "score": 3}
        ]
    },
    {
        "question": "17. Irritability:",
        "options": [
            {"text": "I am no more irritable than usual.", "score": 0},
            {"text": "I am more irritable than usual.", "score": 1},
            {"text": "I am much more irritable than usual.", "score": 2},
            {"text": "I am irritable all the time.", "score": 3}
        ]
    },
    {
        "question": "18. Changes in Appetite:",
        "options": [
            {"text": "I have not experienced any change in my appetite.", "score": 0},
            {"text": "My appetite is somewhat less than usual. / My appetite is somewhat greater than usual.", "score": 1},
            {"text": "My appetite is much less than before. / My appetite is much greater than usual.", "score": 2},
            {"text": "I have no appetite at all. / I crave food all the time.", "score": 3}
        ]
    },
    {
        "question": "19. Concentration Difficulty:",
        "options": [
            {"text": "I can concentrate as well as ever.", "score": 0},
            {"text": "I can't concentrate as well as usual.", "score": 1},
            {"text": "It's hard to keep my mind on anything for very long.", "score": 2},
            {"text": "I find I can't concentrate on anything.", "score": 3}
        ]
    },
    {
        "question": "20. Tiredness or Fatigue:",
        "options": [
            {"text": "I am no more tired or fatigued than usual.", "score": 0},
            {"text": "I get more tired or fatigued more easily than usual.", "score": 1},
            {"text": "I am too tired or fatigued to do a lot of the things I used to do.", "score": 2},
            {"text": "I am too tired or fatigued to do most of the things I used to do.", "score": 3}
        ]
    },
    {
        "question": "21. Loss of Interest in Sex:",
        "options": [
            {"text": "I have not noticed any recent change in my interest in sex.", "score": 0},
            {"text": "I am less interested in sex than I used to be.", "score": 1},
            {"text": "I am much less interested in sex now.", "score": 2},
            {"text": "I have lost interest in sex completely.", "score": 3}
        ]
    }
]

print(f"Defined {len(bdi2_questions)} BDI-II questions.")

In [ ]:

from datetime import datetime, timezone

# bdi2_questions must be defined in the preceding questionnaire cell.
def get_bdi_risk_level(total_score):
    if 0 <= total_score <= 13:
        return "Minimal Depression"
    if 14 <= total_score <= 19:
        return "Mild Depression"
    if 20 <= total_score <= 28:
        return "Moderate Depression"
    if 29 <= total_score <= 63:
        return "Severe Depression"
    raise ValueError("BDI-II score is outside the expected range 0-63.")

# These are two distinct permissions. Saving an assessment does NOT imply
# permission to use it for research/model training.
store_responses = input(
    "Save this sensitive assessment in MongoDB? Type SAVE to consent: "
).strip().upper() == "SAVE"

research_consent = False
if store_responses:
    research_consent = input(
        "Separately, may this de-identified assessment be used for experimental "
        "research/model training? Type RESEARCH to consent, or press Enter to decline: "
    ).strip().upper() == "RESEARCH"

user_responses_for_db = []
raw_selected_responses_list_for_ml = []
total_bdi_score = 0
self_harm_item_flag = False

for q in bdi2_questions:
    print(f"\n{q['question']}")
    options = q["options"]
    while True:
        try:
            choice = int(input(f"Enter a choice from 0 to {len(options) - 1}: "))
        except ValueError:
            print("Enter a whole-number option.")
            continue
        if 0 <= choice < len(options):
            break
        print("That choice is outside the available options.")

    selected_option = options[choice]
    score = int(selected_option["score"])
    user_responses_for_db.append({
        "question": q["question"],
        "selected_text": selected_option["text"],
        "score": score,
    })
    raw_selected_responses_list_for_ml.append(selected_option["text"])
    total_bdi_score += score
    if str(q.get("question", "")).lstrip().startswith("9.") and score > 0:
        self_harm_item_flag = True

risk_level = get_bdi_risk_level(total_bdi_score)
last_inserted_doc_id = None
print("BDI-II score category (screening only, not a diagnosis):", risk_level)

if self_harm_item_flag:
    print(
        "A response to the self-harm item indicates a need for prompt human support. "
        "If there is immediate danger, contact local emergency services or a trusted "
        "person/qualified professional now. Do not rely on this model."
    )

if store_responses:
    survey_data = {
        "total_score": int(total_bdi_score),
        "risk_level": risk_level,
        "responses": user_responses_for_db,
        "raw_responses_for_ml": raw_selected_responses_list_for_ml,
        "storage_consent": True,
        "research_consent": bool(research_consent),
        "training_status": "eligible" if research_consent else "not_consented",
        "created_at": datetime.now(timezone.utc),
        "schema_version": 2,
    }
    client = create_mongo_client()
    try:
        client.admin.command("ping")
        result = client["bdi_result"]["users"].insert_one(survey_data)
        last_inserted_doc_id = result.inserted_id
    finally:
        client.close()
    print("Assessment saved to MongoDB.")
    print("Research/model-training consent:", "granted" if research_consent else "not granted")
    if research_consent:
        print("This record may enter the experimental training queue after eligibility checks.")
else:
    print("No assessment data was stored.")

# Trigger the threshold check after each saved assessment. Training only runs
# when enough newly consented records have accumulated.
if store_responses and research_consent:
    try:
        training_status = maybe_train_from_mongodb()
        print("Automatic training check:", training_status)
    except Exception as exc:
        # A training failure must never undo the user's saved assessment.
        print("Assessment saved; automatic training check did not complete:", repr(exc))


In [ ]:

if not raw_selected_responses_list_for_ml:
    raise RuntimeError("Run the survey cell first to create an in-memory response.")

combined_user_response = " ".join(raw_selected_responses_list_for_ml)
predicted_risk_level = risk_level  # authoritative category remains score-derived

try:
    approved_pipeline, model_metadata = load_approved_experimental_model()
    model_estimate = approved_pipeline.predict(
        [preprocess_text_for_prediction(combined_user_response)]
    )[0]
    print("Experimental text-model estimate:", model_estimate)
    print("Experimental model metadata:", model_metadata)
except RuntimeError:
    print("No approved experimental model is available yet; threshold training has not promoted one.")

print("BDI-II score-derived screening category (not a diagnosis):", predicted_risk_level)


## Example guidance only

The score bands and generated suggestions are not a diagnosis or treatment plan. Do not use this experimental classifier to make health-care decisions; seek help from a qualified professional when needed.

In [ ]:
recommendations_mapping = {
    'Minimal Depression': {
        'main_recommendations': [
            'Maintain current healthy habits.',
            'Continue self-monitoring for any changes in mood or well-being.',
            'Engage in activities that bring joy and relaxation.'
        ],
        'personalized_goals': [
            'Identify new hobbies or interests to pursue.',
            'Strengthen social connections.',
            'Practice daily mindfulness or gratitude.'
        ],
        'lifestyle_advice': {
            'sleep': [
                'Maintain a consistent sleep schedule.',
                'Ensure 7-9 hours of quality sleep per night.',
                'Create a relaxing bedtime routine.'
            ],
            'nutrition': [
                'Eat a balanced diet rich in fruits, vegetables, and whole grains.',
                'Stay hydrated.',
                'Limit processed foods and excessive sugar intake.'
            ],
            'exercise': [
                'Aim for at least 30 minutes of moderate exercise most days of the week.',
                'Incorporate activities you enjoy, like walking, cycling, or dancing.',
                'Consider light strength training.'
            ]
        },
        'motivational_message_templates': [
            'Keep up the great work! Your commitment to well-being is inspiring.',
            'You are doing wonderfully. Continue nurturing your mental health.',
            'Every small step forward is a victory. Celebrate your progress!'
        ]
    },
    'Mild Depression': {
        'main_recommendations': [
            'Consult a mental health professional (therapist or counselor) for a preliminary assessment and guidance.',
            'Increase engagement in positive coping strategies.',
            'Seek support from trusted friends or family members.'
        ],
        'personalized_goals': [
            'Schedule regular therapy sessions.',
            'Identify and challenge negative thought patterns.',
            'Dedicate time each day to self-care activities.'
        ],
        'lifestyle_advice': {
            'sleep': [
                'Prioritize consistent sleep, even on weekends.',
                'Avoid screens before bedtime.',
                'Consider relaxation techniques like deep breathing before sleep.'
            ],
            'nutrition': [
                'Focus on nutrient-dense foods to support brain health.',
                'Avoid skipping meals.',
                'Reduce caffeine and alcohol intake, especially in the evening.'
            ],
            'exercise': [
                'Start with gentle exercise, like daily walks.',
                'Gradually increase intensity and duration as tolerated.',
                'Find a workout buddy for accountability.'
            ]
        },
        'motivational_message_templates': [
            'It takes courage to seek help, and you are taking important steps. Keep going!',
            'Your strength is greater than you know. Healing is a process, and you are not alone.',
            'Believe in your ability to heal. Each effort you make brings you closer to feeling better.'
        ]
    },
    'Moderate Depression': {
        'main_recommendations': [
            'Strongly consider psychotherapy, possibly combined with medication, after consulting a psychiatrist or licensed therapist.',
            'Develop a crisis plan with a support system.',
            'Limit exposure to stressors and create a structured daily routine.'
        ],
        'personalized_goals': [
            'Attend all scheduled therapy and/or medication management appointments.',
            'Practice emotional regulation techniques (e.g., journaling, meditation).',
            'Re-engage in one previously enjoyed activity at a time.'
        ],
        'lifestyle_advice': {
            'sleep': [
                'Aim for 8 hours of uninterrupted sleep.',
                'Consult a doctor if sleep disturbances persist.',
                'Ensure a dark, quiet, and cool sleep environment.'
            ],
            'nutrition': [
                'Maintain regular meal times to stabilize blood sugar.',
                'Consider supplements like Omega-3s or Vitamin D, under medical supervision.',
                'Prepare simple, healthy meals in advance.'
            ],
            'exercise': [
                'Incorporate light to moderate physical activity daily, even if it\'s just stretching.',
                'Engage in outdoor activities to get natural light.',
                'Focus on consistency over intensity.'
            ]
        },
        'motivational_message_templates': [
            'You are resilient, and recovery is possible. Every step, no matter how small, is progress.',
            'Healing is not linear, but your commitment to yourself is powerful. We are here to support you.',
            'Hold onto hope. Even in challenging moments, remember your strength and capacity for change.'
        ]
    },
    'Severe Depression': {
        'main_recommendations': [
            'Seek immediate professional help. Contact a mental health professional, psychiatrist, or emergency services.',
            'Do not try to handle this alone. Reach out to trusted family or friends for support.',
            'Prioritize safety: Remove any means of self-harm from your surroundings.',
            'Consider inpatient or intensive outpatient treatment for stabilization.'
        ],
        'personalized_goals': [
            'Establish immediate contact with a mental health professional or emergency service.',
            'Build a strong support network of family and friends.',
            'Focus on basic self-care: eating, sleeping, and personal hygiene, even minimally.'
        ],
        'lifestyle_advice': {
            'sleep': [
                'Seek medical advice for severe sleep disturbances.',
                'Establish a safe and comfortable sleep environment.',
                'Avoid stimulants like caffeine and nicotine.'
            ],
            'nutrition': [
                'Prioritize regular intake of easily digestible, nutritious foods.',
                'Seek help from others for meal preparation if needed.',
                'Stay hydrated with water.'
            ],
            'exercise': [
                'Engage in very light physical activity as tolerated, such as short walks with supervision.',
                'Focus on gentle movement to improve mood and reduce agitation.',
                'Avoid strenuous activities that might feel overwhelming.'
            ]
        },
        'motivational_message_templates': [
            'Your life is valuable, and there is help available. Please reach out now.',
            'You are not alone in this. Your courage to seek help is a testament to your strength.',
            'Even in the darkest moments, hope and healing are possible. Hold on, help is here.'
        ]
    }
}

print("Recommendation mapping created successfully with granular details.")

In [ ]:
def get_lifestyle_advice(predicted_risk_level, recommendations_mapping):
    """
    Provides specific lifestyle advice based on the predicted depression risk level.

    Args:
        predicted_risk_level (str): The predicted depression risk level.
        recommendations_mapping (dict): A dictionary containing recommendations,
                                        goals, and lifestyle advice for each risk level.

    Returns:
        dict: A dictionary of lifestyle advice categories (e.g., 'sleep', 'nutrition', 'exercise')
              and their corresponding tips, or an empty dictionary if the level is not found.
    """
    if predicted_risk_level in recommendations_mapping:
        return recommendations_mapping[predicted_risk_level].get('lifestyle_advice', {})
    else:
        return {}

# Example usage with the previously predicted risk level
predicted_risk_level_for_advice = predicted_risk_level # Using the output from the previous cell
lifestyle_advice_dict = get_lifestyle_advice(predicted_risk_level_for_advice, recommendations_mapping)

print(f"\nLifestyle Advice for '{predicted_risk_level_for_advice}':")
if lifestyle_advice_dict:
    for category, tips in lifestyle_advice_dict.items():
        print(f"  {category.capitalize()}:")
        for tip in tips:
            print(f"    - {tip}")
else:
    print("  No specific lifestyle advice found for this risk level.")

In [ ]:
import random

def generate_motivational_message(predicted_risk_level, recommendations_mapping):
    """
    Generates a motivational message based on the predicted depression risk level.

    Args:
        predicted_risk_level (str): The predicted depression risk level.
        recommendations_mapping (dict): A dictionary containing recommendations,
                                        goals, and lifestyle advice for each risk level.

    Returns:
        str: A randomly selected motivational message or a default message.
    """
    if predicted_risk_level in recommendations_mapping:
        messages = recommendations_mapping[predicted_risk_level].get('motivational_message_templates', [])
        if messages:
            return random.choice(messages)
    return "Keep pushing forward, every step counts on your journey to well-being!"

# Example usage with the previously predicted risk level
predicted_risk_level_for_message = predicted_risk_level # Using the output from the previous cell
motivational_message = generate_motivational_message(predicted_risk_level_for_message, recommendations_mapping)

print(f"\nMotivational Message for '{predicted_risk_level_for_message}':")
print(motivational_message)

In [ ]:
import random

def suggest_goals(predicted_risk_level, recommendations_mapping):
    """
    Suggests personalized goals based on the predicted depression risk level.

    Args:
        predicted_risk_level (str): The predicted depression risk level.
        recommendations_mapping (dict): A dictionary containing recommendations,
                                        goals, and lifestyle advice for each risk level.

    Returns:
        list: A list of suggested goals, or an empty list if the level is not found.
    """
    if predicted_risk_level in recommendations_mapping:
        return recommendations_mapping[predicted_risk_level].get('personalized_goals', [])
    else:
        return []

def generate_personalized_response(predicted_risk_level, recommendations_mapping):
    """
    Generates a comprehensive, personalized textual response for the user
    based on their predicted depression risk level, integrating recommendations,
    goals, lifestyle tips, and motivational elements.

    Args:
        predicted_risk_level (str): The predicted depression risk level.
        recommendations_mapping (dict): A dictionary containing recommendations,
                                        goals, and lifestyle advice for each risk level.

    Returns:
        str: A comprehensive personalized textual response.
    """
    response_parts = []

    # 1. Main Recommendations
    main_recs = recommendations_mapping.get(predicted_risk_level, {}).get('main_recommendations', [])
    if main_recs:
        response_parts.append(f"Based on your assessment, your predicted depression risk level is: {predicted_risk_level}.\n")
        response_parts.append("Here are some immediate recommendations for you:")
        for i, rec in enumerate(main_recs):
            response_parts.append(f"  - {rec}")
        response_parts.append("\n")

    # 2. Personalized Goals
    goals = suggest_goals(predicted_risk_level, recommendations_mapping)
    if goals:
        response_parts.append("To help you on your journey, consider these personalized goals:")
        for i, goal in enumerate(goals):
            response_parts.append(f"  - {goal}")
        response_parts.append("\n")

    # 3. Lifestyle Advice
    lifestyle_advice = get_lifestyle_advice(predicted_risk_level, recommendations_mapping)
    if lifestyle_advice:
        response_parts.append("Here is some tailored lifestyle advice:")
        for category, tips in lifestyle_advice.items():
            response_parts.append(f"  {category.capitalize()} Tips:")
            for tip in tips:
                response_parts.append(f"    - {tip}")
        response_parts.append("\n")

    # 4. Motivational Message
    motivational_message = generate_motivational_message(predicted_risk_level, recommendations_mapping)
    if motivational_message:
        response_parts.append("A message of encouragement for you:")
        response_parts.append(f"\" {motivational_message} \"\n")

    if not response_parts:
        return "Could not generate a personalized response. Please check the predicted risk level."

    return '\n'.join(response_parts)

# Demonstrate the usage of the new function
final_personalized_response = generate_personalized_response(predicted_risk_level, recommendations_mapping)
print("--- Comprehensive Personalized Response ---")
print(final_personalized_response)

In [ ]:
from datetime import datetime, timezone

if last_inserted_doc_id is None:
    print("No consented MongoDB record is available to update.")
elif input("Also store the generated example response in MongoDB? Type UPDATE to confirm: ").strip().upper() != "UPDATE":
    print("Response update skipped.")
else:
    client = create_mongo_client()
    try:
        update_result = client["bdi_result"]["users"].update_one(
            {"_id": last_inserted_doc_id, "research_consent": True},
            {
                "$set": {
                    "final_personalized_response": final_personalized_response,
                    "updated_at": datetime.now(timezone.utc),
                }
            },
        )
        if update_result.matched_count != 1:
            raise RuntimeError("The consented survey record was not found for update.")
    finally:
        client.close()
    print("Generated example response stored with explicit confirmation.")